###1. Basic Tasks

**1. Install and authenticate the Databricks CLI using OAuth U2M against your workspace.**

**i. Install Databricks CLI**
- brew tap databricks/tap
- brew trust databricks/tap
- brew install databricks

**ii. Authenticate using OAuth U2M**
- databricks auth login --host https://dbc-xxxxxxxx-xxxx.cloud.databricks.com

**iii. Give the profile a name**
- surajit

**iv. Verify authentication**
- databricks auth profiles
- databricks current-user me -p surajit

**2. Initialize a Declarative Automation Bundle project (databricks bundle init, or by hand) with one job resource.**

**1. Create a Bundle project**
- cd ~/downloads
- databricks bundle init
- default-python
- surajit_dab

**2. Go into the project**
- cd surajit_dab

**3. Define one Job resource**
- resources/sample_job.job.yml
```resources:
resources:
  jobs:
    dev_<email>_sample_job:
      name: "[dev <email>] sample_job"
      trigger:
        pause_status: PAUSED
        periodic:
          interval: 1
          unit: DAYS
      max_concurrent_runs: 4
      tasks:
        - task_key: notebook_task
          notebook_task:
            notebook_path: /Workspace/Users/<email>/.bundle/surajit_dabs/dev/files/src/sample_notebook
            source: WORKSPACE
        - task_key: refresh_pipeline
          depends_on:
            - task_key: notebook_task
          pipeline_task:
            pipeline_id: 6462ff3b-0a06-4211-ac0b-4bbfdddcad82
      tags:
        dev: <email>
      queue:
        enabled: true
      parameters:
        - name: catalog
          default: dev
        - name: schema
          default: bronze
      environments:
        - environment_key: default
          spec:
            environment_version: "5"
```

**4. Validate the Bundle/script**
- databricks bundle validate

**5. Deploy the Bundle**
- databricks bundle deploy -t dev

**6. Run the Job**
- databricks bundle run dev_<-email>_sample_job -t dev

**3. Run databricks bundle validate and databricks bundle deploy -t dev, then confirm the job appears in your workspace.**

**Step 1: Navigate to my bundle project**
- cd <-my-bundle-project>

**Step 2: Validate the bundle**
- databricks bundle validate -t dev

**Step 3: Deploy the bundle to the dev target**
- databricks bundle deploy -t dev

**Step 4: Confirm the job exists**
- Now I be able to see the job dev_<-email>_sample_job in my workspace job & pipelines that was defined in bundle

###2. Intermediate Tasks

**4. Add a second target (staging or prod) to your databricks.yml with a different workspace host and
run_as service principal, and deploy to it.**

I configured a prod target in databricks.yml that points to a different Databricks workspace and uses a service principal as the runtime identity.

**1. Configure the prod target**
```
targets:
  dev:
    mode: development
    default: true

    workspace:
      host: https://<DEV-WORKSPACE-URL>

  prod:
    mode: production

    workspace:
      host: https://<PROD-WORKSPACE-URL>
      root_path: /Workspace/Users/<DEPLOYER-EMAIL>/.bundle/${bundle.name}/${bundle.target}

    variables:
      catalog: prod
      schema: bronze

    permissions:
      - user_name: <DEPLOYER-EMAIL>
        level: CAN_MANAGE

    run_as:
      service_principal_name: <SERVICE-PRINCIPAL-APPLICATION-ID>
```
In my setup:
- The DEV target points to the development workspace.
- The PROD target points to a different production workspace.
- The deployment user authenticates to the production workspace using OAuth U2M.
- The run_as service principal is used as the runtime identity for the production Job/Pipeline.
- The service principal was granted the required permissions, including access to the prod catalog and prod.bronze schema.

**2. Authenticate to the production workspace**

I created a separate CLI profile for the production workspace:
- databricks auth login --host https://<PROD-WORKSPACE-URL>

I then verified the authenticated identity:
- databricks current-user me --profile prod-developer

**3. Validate the production target**
- databricks bundle validate -t prod --profile prod-developer

The result was:
- Validation OK!

**4. Deploy to production**
- databricks bundle deploy -t prod --profile prod-developer

The bundle successfully created the resources:
- Created jobs.sample_job
- Created jobs.sample_job.permissions
- Created pipelines.surajit_dabs_etl
- Created pipelines.surajit_dabs_etl.permissions

Files: 24 uploaded, 0 deleted

Resources: 4 created, 0 changed, 0 deleted, 0 unchanged

**5. Run the deployed Job**
- databricks bundle run sample_job -t prod --profile prod-developer

**5. Configure M2M (service principal) authentication for the CLI and use it instead of your personal U2M login for a deploy command.**

I configured Machine-to-Machine (M2M) authentication for the Databricks CLI using the existing production Service Principal instead of my personal OAuth U2M account.
1. Service Principal: prod
- Client ID: d7f7ae0c-376f-4f1f-8586-0c41fd8c43d3

I generated a client secret for this Service Principal from the Databricks workspace.
2. Configure the M2M CLI profile

I added a separate prod-m2m profile to:
- ~/.databrickscfg

Configuration:
```
[prod-m2m]
host = https://<PROD-WORKSPACE-URL>
client_id = <CLIENT_ID/SERVICE_PRINCIPLE_NAME>
client_secret = <CLIENT_SECRET>
```
3. Verify M2M authentication

I verified the profile using:
- databricks auth profiles

The prod-m2m profile was successfully recognized.

I then ran:
- databricks current-user me --profile prod-m2m

The CLI identified the Service Principal:
- User: <CLIENT_ID/SERVICE_PRINCIPLE_NAME>

This confirmed that the CLI was authenticated using M2M Service Principal authentication, rather than my personal U2M account.

4. Deploy the existing bundle using M2M

I used the same DAB project from the previous task and validated it with:
- databricks bundle validate -t prod --profile prod-m2m

After resolving the required permissions for the Service Principal on the existing job, I deployed the bundle using:
- databricks bundle deploy -t prod --profile prod-m2m

The deployment completed successfully.

**6. Write a GitHub Actions workflow that runs databricks bundle validate on every pull request, without deploying anything.**

I created a GitHub Actions workflow at .github/workflows/databricks-validate.yml that is triggered whenever a Pull Request is opened or updated. The workflow checks out the repository, installs the Databricks CLI, authenticates using Databricks Service Principal credentials stored as GitHub Secrets, and runs:
- databricks bundle validate -t prod

No databricks bundle deploy command is included, so the workflow only validates the bundle and does not deploy or modify any Databricks resources.

```
name: Databricks Bundle Validation

on:
  pull_request:

jobs:
  validate:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Install Databricks CLI
        uses: databricks/setup-cli@main

      - name: Validate Databricks Bundle
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
          DATABRICKS_CLIENT_ID: ${{ secrets.DATABRICKS_CLIENT_ID }}
          DATABRICKS_CLIENT_SECRET: ${{ secrets.DATABRICKS_CLIENT_SECRET }}
        run: |
          databricks bundle validate -t prod
```
This provides an automated validation check for every Pull Request before changes are merged.

### 3. Advanced Tasks

**7. Extend the GitHub Actions workflow to deploy to prod on merge to main using OIDC authentication
(no stored secrets), including the correct permissions: id-token: write block.**

I extended the GitHub Actions workflow so that Pull Requests only run `databricks bundle validate`, while a merge to the main branch triggers a production deployment.

The production deployment uses GitHub Actions OIDC authentication with the Databricks Service Principal. No Databricks client secret is stored in GitHub. The workflow grants the required OIDC permission using:
```
permissions:
  id-token: write
  contents: read
```
The workflow is:
```
name: Databricks CI/CD

on:
  pull_request:
  push:
    branches:
      - main

jobs:
  validate:
    name: Validate Databricks Bundle
    if: github.event_name == 'pull_request'
    runs-on: ubuntu-latest

    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Install Databricks CLI
        uses: databricks/setup-cli@main

      - name: Validate bundle
        env:
          DATABRICKS_HOST: <DATABRICKS_HOST_URL>
        run: |
          databricks bundle validate -t prod

  deploy:
    name: Deploy to Production
    if: github.event_name == 'push' && github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest

    permissions:
      id-token: write
      contents: read

    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Install Databricks CLI
        uses: databricks/setup-cli@main

      - name: Deploy bundle to production
        env:
          DATABRICKS_HOST: <DATABRICKS_HOST_URL>
          DATABRICKS_CLIENT_ID: <DATABRICKS_CILENT_ID_OF_HOST>
        run: |
          databricks bundle deploy -t prod
```
The workflow therefore provides a CI/CD flow where Pull Requests are validated automatically, and changes merged into main are deployed to the production Databricks workspace using OIDC-based, secretless authentication.

**8. Design a rollback plan: if a bundle deploy to prod breaks a job, what CLI commands would you run to redeploy the previous working version quickly?**

If a production bundle deployment introduces a breaking change to a job, the safest approach is to redeploy the last known-good Git version of the bundle.

Since the Databricks Bundle is stored in Git, the rollback can be performed by checking out the previous working commit and deploying that version.

1. Identify the previous working commit
Check the Git history:
- git log --oneline
For example:
```
a81f23c Fix production pipeline
b72d91a Add new transformation
c61e452 Initial production deployment
```
If a81f23c is the current broken deployment and b72d91a was the last known-good version, use b72d91a.

2. Check out the previous working version
- git checkout b72d91a

Verify the bundle:
- databricks bundle validate -t prod --profile prod-m2m

3. Redeploy the previous version

Using the M2M Service Principal authentication configured earlier:
- databricks bundle deploy -t prod --profile prod-m2m

This redeploys the bundle configuration and code from the previous working Git commit.

4. Verify the job

After deployment, run the production job:
- databricks bundle run sample_job -t prod --profile prod-m2m

Then verify that the job completes successfully in the Databricks workspace.

**9. Write a one-page onboarding guide for a new team member explaining how a change moves from a
local databricks.yml edit to running safely in production, referencing the CLI, the bundle lifecycle, and the CI/CD workflow together.**

**Onboarding Guide: From Local Change to Production**

Welcome to the Databricks project. We use **Databricks Declarative Automation Bundles (DABs)**, the **Databricks CLI**, Git, and **GitHub Actions** to move changes safely from development into production.

The overall flow is:

```text
Local Development
      ↓
Edit databricks.yml / source files
      ↓
Bundle Validate
      ↓
Git Branch + Commit
      ↓
Pull Request
      ↓
GitHub Actions → Bundle Validate
      ↓
Merge to main
      ↓
GitHub Actions → OIDC Authentication
      ↓
Bundle Deploy to PROD
      ↓
Run / Monitor Production Job
```

---

**1. Set up Databricks CLI Authentication**

Install the Databricks CLI on your Mac:

```bash
brew tap databricks/tap
brew trust databricks/tap
brew install databricks
```

For local development, authenticate using your personal **OAuth U2M** account:

```bash
databricks auth login --host <DEV_WORKSPACE_URL>
```

Verify the authenticated user:

```bash
databricks current-user me
```

For production deployments, CI/CD uses **M2M/OIDC authentication** with the production Service Principal rather than a developer's personal credentials.

---

**2. Make a Local Change**

Clone the repository and create a feature branch:

```bash
git checkout -b feature/my-change
```

Make the required changes to files such as:

```text
databricks.yml
src/
resources/
```

For example, if a job configuration needs to change, modify the relevant resource in the bundle configuration.

The `databricks.yml` file defines the bundle and its targets, such as:

```yaml
targets:
  dev:
    ...

  prod:
    mode: production
    workspace:
      host: https://dbc-af5175d7-cbe7.cloud.databricks.com/
    run_as:
      service_principal_name: d7f7ae0c-376f-4f1f-8586-0c41fd8c43d3
```

---

**3. Understand the Bundle Lifecycle**

A Databricks Bundle generally follows this lifecycle:

```text
Validate → Deploy → Run
```

**Validate**

First check whether the bundle configuration is valid:

```bash
databricks bundle validate -t dev
```

For production:

```bash
databricks bundle validate -t prod
```

Validation checks the bundle configuration without deploying the resources.

**Deploy**

When ready to deploy:

```bash
databricks bundle deploy -t dev
```

The CLI synchronizes the bundle's files and resources with the target Databricks workspace.

For production, deployment is normally handled by GitHub Actions rather than manually from a developer's machine.

**Run**

After deployment, a job can be run using:

```bash
databricks bundle run sample_job -t dev
```

This allows the developer to verify the change in the development environment.

---

**4. Commit and Create a Pull Request**

Once the change works in development:

```bash
git add .
git commit -m "Update sample job"
git push origin feature/my-change
```

Create a Pull Request from the feature branch into `main`.

---

**5. CI: Automatic Validation**

Every Pull Request automatically triggers the GitHub Actions validation workflow.

The workflow runs:

```bash
databricks bundle validate -t prod
```

It **does not deploy anything**.

The process is:

```text
Pull Request
     ↓
GitHub Actions
     ↓
Install Databricks CLI
     ↓
Authenticate
     ↓
databricks bundle validate
     ↓
Validation successful
```

If validation fails, fix the problem locally, commit the changes, and push again. The workflow will run again automatically.

---

**6. Merge to Main → Production Deployment**

Once the Pull Request has been reviewed and approved, merge it into `main`.

A push to `main` triggers the production deployment workflow.

GitHub Actions authenticates to Databricks using **OIDC**:

```yaml
permissions:
  id-token: write
  contents: read
```

The workflow then executes:

```bash
databricks bundle deploy -t prod
```

There is **no stored Databricks client secret** in GitHub.

The authentication flow is:

```text
GitHub Actions
      ↓
OIDC Token
      ↓
Databricks
      ↓
Production Service Principal
      ↓
Production Workspace
```

The production target uses the Service Principal as the runtime identity:

```yaml
run_as:
  service_principal_name: <SERVICE_PRINCIPLE_OF_HOST>
```

---

**7. Verify Production**

After the deployment completes, verify the production job:

```bash
databricks bundle run sample_job -t prod --profile prod-m2m
```

Also check the job and pipeline in the Databricks workspace for successful execution.

---

**8. Rollback if Something Goes Wrong**

If a production deployment breaks a job, identify the last known-good Git commit:

```bash
git log --oneline
```

Check it out:

```bash
git checkout <LAST_WORKING_COMMIT>
```

Validate and redeploy:

```bash
databricks bundle validate -t prod --profile prod-m2m

databricks bundle deploy -t prod --profile prod-m2m
```

Then verify the job:

```bash
databricks bundle run sample_job -t prod --profile prod-m2m
```

Afterward, fix the issue in a new branch and go through the normal Pull Request process.

---

**Quick Reference**

| Stage      | Action             | Command/Process                           |
| ---------- | ------------------ | ----------------------------------------- |
| Local      | Edit bundle/source | `databricks.yml`, `src/`, `resources/`    |
| Local      | Validate           | `databricks bundle validate -t dev`       |
| Local      | Deploy to Dev      | `databricks bundle deploy -t dev`         |
| Local      | Test               | `databricks bundle run sample_job -t dev` |
| Git        | Commit             | `git commit`                              |
| PR         | CI validation      | `databricks bundle validate -t prod`      |
| Merge      | Production CI/CD   | GitHub Actions                            |
| Production | Authentication     | GitHub OIDC                               |
| Production | Deploy             | `databricks bundle deploy -t prod`        |
| Production | Verify             | Run/monitor job                           |
| Emergency  | Rollback           | Checkout previous Git commit + redeploy   |

**Golden Rule**

**Never treat a production deployment as a manual file upload.** Make the change locally, validate and test it in development, raise a Pull Request, let CI validate it, and only deploy to production after the change is merged to `main`. This gives us a repeatable and auditable path from **code → bundle → CI validation → production deployment**.
